# Lab 02 - Ti?n x? l? d? li?u WDI chung cho nh?m 15

Notebook duy nh?t n?y thay th? c?c notebook ri?ng l? tr??c ??y. Quy tr?nh ??ng theo y?u c?u b?i t?p: d?ng Python ?? ti?n x? l? d? li?u t? `WDIEXCEL.xlsx`, t?o m?t file d? li?u s?ch chung, sau ?? tr?ch c?c file CSV theo t?ng m?c ti?u ?? k?t n?i v?o Tableau.

C?c b??c ch?nh:

1. ??c `WDIEXCEL.xlsx` t? th? m?c g?c d? ?n.
2. Gi? c?c qu?c gia/v?ng l?nh th? c? `Region` h?p l? trong sheet `Country`, lo?i c?c d?ng t?ng h?p nh? World ho?c nh?m thu nh?p.
3. L?c to?n b? indicator d?ng cho 4 m?c ti?u: kinh t?, gi?o d?c, s?c kh?e v? m?i tr??ng.
4. L?m s?ch ? c?p `Country_Code` - `Indicator_Code` cho giai ?o?n 2000-2023.
5. Xu?t `data_output/wdi_clean_base.csv` ? d?ng long format.
6. T? file s?ch chung, pivot ra 4 file m?c ti?u: `wdi_economy.csv`, `wdi_education.csv`, `wdi_health.csv`, `wdi_environment.csv`.


## Quy t?c l?m s?ch chung

- Ch? gi? c?c ??n v? ??a l? c? `Region` h?p l? trong metadata WDI.
- Ch? x?t giai ?o?n 2000-2023.
- N?u m?t c?p qu?c gia-indicator kh?ng c? quan s?t n?o, lo?i kh?i d? li?u s?ch.
- N?u kho?ng thi?u n?i b? d?i h?n 2 n?m li?n ti?p, lo?i c?p qu?c gia-indicator ?? ?? tr?nh t?o chu?i d? li?u gi?.
- N?u kho?ng thi?u n?i b? d?i t?i ?a 2 n?m, n?i suy tuy?n t?nh.
- Kh?ng ngo?i suy ? ??u ho?c cu?i chu?i th?i gian.

File `wdi_clean_base.csv` l? ngu?n s?ch chung. C?c file theo m?c ti?u trong `data_output` ???c tr?ch t? file n?y, kh?ng ??c tr?c ti?p t? Excel ri?ng l? n?a.


## File ??u ra

- `data_output/wdi_clean_base.csv`: d? li?u WDI ?? ti?n x? l? chung, long format.
- `data_output/wdi_economy.csv`: d? li?u m?c ti?u kinh t?, wide format cho Tableau.
- `data_output/wdi_education.csv`: d? li?u m?c ti?u gi?o d?c, wide format cho Tableau.
- `data_output/wdi_health.csv`: d? li?u m?c ti?u s?c kh?e v? d?n s?, wide format cho Tableau.
- `data_output/wdi_environment.csv`: d? li?u m?c ti?u m?i tr??ng, wide format cho Tableau.
- `data_output/wdi_indicator_audit.csv`: ki?m tra indicator c? trong WDI v? metadata li?n quan.
- `data_output/wdi_cleaning_report.csv`: b?o c?o tr?ng th?i l?m s?ch cho t?ng c?p qu?c gia-indicator.


In [ ]:

from __future__ import annotations

import csv
import math
import zipfile
from pathlib import Path
from xml.etree import ElementTree as ET

ROOT = Path.cwd()
if not (ROOT / "WDIEXCEL.xlsx").exists() and (ROOT.parent / "WDIEXCEL.xlsx").exists():
    ROOT = ROOT.parent

XLSX_PATH = ROOT / "WDIEXCEL.xlsx"
OUTPUT_DIR = ROOT / "data_output"
YEARS = [str(y) for y in range(2000, 2024)]

OBJECTIVES = {
    "economy": {
        "output": "wdi_economy.csv",
        "label": "Kinh t?",
        "indicators": {
            "NY.GDP.MKTP.CD": {
                "clean_name": "GDP_current_USD",
                "unit": "US$",
                "role": "Quy m? n?n kinh t?"
            },
            "NY.GDP.PCAP.CD": {
                "clean_name": "GDP_per_capita_current_USD",
                "unit": "US$ per person",
                "role": "M?c s?ng b?nh qu?n"
            },
            "NY.GDP.MKTP.KD.ZG": {
                "clean_name": "GDP_growth_annual_pct",
                "unit": "%",
                "role": "T?c ?? t?ng tr??ng"
            },
            "SI.POV.DDAY": {
                "clean_name": "Poverty_headcount_3usd_2021PPP_pct",
                "unit": "%",
                "role": "T? l? ngh?o"
            }
        }
    },
    "education": {
        "output": "wdi_education.csv",
        "label": "Gi?o d?c",
        "indicators": {
            "SE.PRM.NENR": {
                "clean_name": "Primary_Net_Enrollment",
                "unit": "%",
                "role": "M?c ?? ti?p c?n gi?o d?c ti?u h?c"
            },
            "SE.SEC.NENR": {
                "clean_name": "Secondary_Net_Enrollment",
                "unit": "%",
                "role": "M?c ?? ti?p c?n gi?o d?c trung h?c"
            },
            "SE.ADT.LITR.ZS": {
                "clean_name": "Adult_Literacy_Rate",
                "unit": "%",
                "role": "K?t qu? gi?o d?c qua t? l? bi?t ch?"
            },
            "SE.XPD.TOTL.GD.ZS": {
                "clean_name": "Education_Expenditure_GDP",
                "unit": "% of GDP",
                "role": "M?c ??u t? c?a ch?nh ph? cho gi?o d?c"
            }
        }
    },
    "health": {
        "output": "wdi_health.csv",
        "label": "S?c kh?e v? d?n s?",
        "indicators": {
            "SP.DYN.LE00.IN": {
                "clean_name": "Life_expectancy",
                "unit": "years",
                "role": "Tu?i th? trung b?nh"
            },
            "SH.DYN.MORT": {
                "clean_name": "Under5_mortality",
                "unit": "per 1,000 live births",
                "role": "T? l? t? vong tr? em d??i 5 tu?i"
            },
            "SH.XPD.CHEX.GD.ZS": {
                "clean_name": "Health_expenditure_pct_GDP",
                "unit": "% of GDP",
                "role": "Chi ti?u y t?"
            },
            "SP.POP.TOTL": {
                "clean_name": "Population",
                "unit": "people",
                "role": "Quy m? d?n s?"
            }
        }
    },
    "environment": {
        "output": "wdi_environment.csv",
        "label": "M?i tr??ng",
        "indicators": {
            "EN.GHG.CO2.MT.CE.AR5": {
                "clean_name": "CO2_Emissions_Total_MtCO2e",
                "unit": "Mt CO2e",
                "role": "Quy m? ph?t th?i CO2"
            },
            "EN.GHG.CO2.PC.CE.AR5": {
                "clean_name": "CO2_Emissions_Per_Capita",
                "unit": "t CO2e/capita",
                "role": "Ph?t th?i CO2 b?nh qu?n ??u ng??i"
            },
            "EG.FEC.RNEW.ZS": {
                "clean_name": "Renewable_Energy_Pct",
                "unit": "%",
                "role": "T? tr?ng n?ng l??ng t?i t?o"
            },
            "AG.LND.FRST.ZS": {
                "clean_name": "Forest_Area_Pct",
                "unit": "% of land area",
                "role": "T? l? di?n t?ch r?ng"
            }
        }
    }
}
ALL_INDICATORS = {
    code: {**meta, "objective": objective, "objective_label": spec["label"]}
    for objective, spec in OBJECTIVES.items()
    for code, meta in spec["indicators"].items()
}

NS = {
    "main": "http://schemas.openxmlformats.org/spreadsheetml/2006/main",
    "rel": "http://schemas.openxmlformats.org/officeDocument/2006/relationships",
    "pkgrel": "http://schemas.openxmlformats.org/package/2006/relationships",
}


def col_index(cell_ref: str) -> int:
    letters = "".join(ch for ch in cell_ref if ch.isalpha())
    value = 0
    for ch in letters:
        value = value * 26 + ord(ch.upper()) - 64
    return value - 1


def load_shared_strings(zf: zipfile.ZipFile) -> list[str]:
    if "xl/sharedStrings.xml" not in zf.namelist():
        return []
    values: list[str] = []
    with zf.open("xl/sharedStrings.xml") as fh:
        for _, elem in ET.iterparse(fh, events=("end",)):
            if elem.tag.endswith("}si"):
                texts = [node.text or "" for node in elem.iter() if node.tag.endswith("}t")]
                values.append("".join(texts))
                elem.clear()
    return values


def get_sheet_paths(zf: zipfile.ZipFile) -> dict[str, str]:
    workbook = ET.fromstring(zf.read("xl/workbook.xml"))
    rels = ET.fromstring(zf.read("xl/_rels/workbook.xml.rels"))
    rel_map = {rel.attrib["Id"]: rel.attrib["Target"] for rel in rels.findall("pkgrel:Relationship", NS)}
    paths: dict[str, str] = {}
    for sheet in workbook.findall("main:sheets/main:sheet", NS):
        name = sheet.attrib["name"]
        rel_id = sheet.attrib[f"{{{NS['rel']}}}id"]
        target = rel_map[rel_id].replace("\\", "/")
        if not target.startswith("xl/"):
            target = "xl/" + target
        paths[name] = target
    return paths


def cell_value(cell: ET.Element, shared: list[str]):
    cell_type = cell.attrib.get("t")
    if cell_type == "inlineStr":
        return "".join(node.text or "" for node in cell.iter() if node.tag.endswith("}t"))
    value = cell.find("main:v", NS)
    if value is None or value.text is None:
        return None
    raw = value.text
    if cell_type == "s":
        return shared[int(raw)]
    if cell_type == "b":
        return raw == "1"
    try:
        return float(raw)
    except ValueError:
        return raw


def iter_rows(zf: zipfile.ZipFile, sheet_path: str, shared: list[str]):
    with zf.open(sheet_path) as fh:
        for _, row in ET.iterparse(fh, events=("end",)):
            if not row.tag.endswith("}row"):
                continue
            values = {}
            current_index = -1
            for cell in row.findall("main:c", NS):
                ref = cell.attrib.get("r", "")
                index = col_index(ref) if ref else current_index + 1
                current_index = index
                values[index] = cell_value(cell, shared)
            if values:
                max_col = max(values)
                yield [values.get(i) for i in range(max_col + 1)]
            row.clear()


def read_small_sheet(zf: zipfile.ZipFile, path: str, shared: list[str]) -> list[dict[str, object]]:
    rows = iter_rows(zf, path, shared)
    header = [str(v) if v is not None else "" for v in next(rows)]
    out = []
    for row in rows:
        out.append({header[i]: row[i] if i < len(row) else None for i in range(len(header))})
    return out


def as_float(value):
    if value is None or value == "":
        return None
    try:
        number = float(value)
    except (TypeError, ValueError):
        return None
    if math.isnan(number):
        return None
    return number


def longest_internal_gap(values: list[float | None]) -> int:
    valid = [idx for idx, value in enumerate(values) if value is not None]
    if not valid:
        return len(values)
    start, end = min(valid), max(valid)
    best = current = 0
    for value in values[start : end + 1]:
        if value is None:
            current += 1
            best = max(best, current)
        else:
            current = 0
    return best


def interpolate_short_gaps(values: list[float | None]) -> tuple[list[float | None], set[int]]:
    cleaned = values[:]
    filled_indices: set[int] = set()
    valid = [idx for idx, value in enumerate(cleaned) if value is not None]
    if len(valid) < 2:
        return cleaned, filled_indices
    for left, right in zip(valid, valid[1:]):
        gap = right - left - 1
        if 0 < gap <= 2:
            start, end = cleaned[left], cleaned[right]
            assert start is not None and end is not None
            step = (end - start) / (right - left)
            for offset in range(1, gap + 1):
                idx = left + offset
                cleaned[idx] = start + step * offset
                filled_indices.add(idx)
    return cleaned, filled_indices


def clean_country_indicator(values: list[float | None]) -> tuple[list[float | None] | None, str, set[int]]:
    observed = sum(value is not None for value in values)
    if observed == 0:
        return None, "dropped_no_observation", set()
    if longest_internal_gap(values) > 2:
        return None, "dropped_internal_gap_gt_2_years", set()
    cleaned, filled_indices = interpolate_short_gaps(values)
    return cleaned, "kept", filled_indices


def write_csv(path: Path, rows: list[dict], fieldnames: list[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8-sig") as fh:
        writer = csv.DictWriter(fh, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()
        for row in rows:
            writer.writerow(row)


def build_base_dataset() -> tuple[list[dict], list[dict], list[dict]]:
    if not XLSX_PATH.exists():
        raise FileNotFoundError(f"Missing source file: {XLSX_PATH}")
    with zipfile.ZipFile(XLSX_PATH) as zf:
        shared = load_shared_strings(zf)
        paths = get_sheet_paths(zf)
        series_rows = read_small_sheet(zf, paths["Series"], shared)
        country_rows = read_small_sheet(zf, paths["Country"], shared)

        real_countries = {
            str(row["Country Code"]): {
                "Country_Name": row.get("Table Name") or row.get("Short Name") or row.get("Country Code"),
                "Country_Code": row.get("Country Code"),
                "Region": row.get("Region"),
                "Income_Group": row.get("Income Group"),
            }
            for row in country_rows
            if row.get("Region") not in (None, "")
        }
        series_by_code = {str(row["Series Code"]): row for row in series_rows}

        rows = iter_rows(zf, paths["Data"], shared)
        header = [str(v) if v is not None else "" for v in next(rows)]
        idx = {name: header.index(name) for name in ["Country Code", "Indicator Code"]}
        year_idx = {year: header.index(year) for year in YEARS if year in header}

        raw: dict[tuple[str, str], list[float | None]] = {}
        for row in rows:
            indicator_code = str(row[idx["Indicator Code"]]) if idx["Indicator Code"] < len(row) and row[idx["Indicator Code"]] is not None else ""
            if indicator_code not in ALL_INDICATORS:
                continue
            country_code = str(row[idx["Country Code"]]) if idx["Country Code"] < len(row) and row[idx["Country Code"]] is not None else ""
            if country_code not in real_countries:
                continue
            raw[(country_code, indicator_code)] = [as_float(row[year_idx[year]]) if year_idx[year] < len(row) else None for year in YEARS]

    audit_rows = []
    for code, meta in ALL_INDICATORS.items():
        series = series_by_code.get(code, {})
        audit_rows.append({
            "Objective": meta["objective"],
            "Objective_Label": meta["objective_label"],
            "Indicator_Code": code,
            "Clean_Name": meta["clean_name"],
            "Confirmed_In_WDI": "Yes" if code in series_by_code else "No",
            "Indicator_Name": series.get("Indicator Name", ""),
            "Topic": series.get("Topic", ""),
            "Unit": meta.get("unit", ""),
            "Analysis_Role": meta.get("role", ""),
        })

    base_rows = []
    cleaning_rows = []
    for country_code, country in sorted(real_countries.items()):
        for indicator_code, meta in ALL_INDICATORS.items():
            values = raw.get((country_code, indicator_code), [None] * len(YEARS))
            cleaned, status, filled_indices = clean_country_indicator(values)
            cleaning_rows.append({
                "Country_Code": country_code,
                "Country_Name": country["Country_Name"],
                "Objective": meta["objective"],
                "Indicator_Code": indicator_code,
                "Clean_Name": meta["clean_name"],
                "Observed_Values": sum(value is not None for value in values),
                "Internal_Max_Missing_Gap": longest_internal_gap(values),
                "Interpolated_Values": len(filled_indices),
                "Status": status,
            })
            if cleaned is None:
                continue
            series = series_by_code.get(indicator_code, {})
            for i, (year, value) in enumerate(zip(YEARS, cleaned)):
                if value is None:
                    continue
                base_rows.append({
                    **country,
                    "Year": int(year),
                    "Objective": meta["objective"],
                    "Objective_Label": meta["objective_label"],
                    "Indicator_Code": indicator_code,
                    "Indicator_Name": series.get("Indicator Name", ""),
                    "Clean_Name": meta["clean_name"],
                    "Unit": meta.get("unit", ""),
                    "Value": value,
                    "Was_Interpolated": "Yes" if i in filled_indices else "No",
                })
    base_rows.sort(key=lambda r: (r["Objective"], r["Country_Name"], r["Indicator_Code"], r["Year"]))
    return base_rows, audit_rows, cleaning_rows


def build_objective_outputs(base_rows: list[dict]) -> dict[str, int]:
    counts = {}
    for objective, spec in OBJECTIVES.items():
        metric_order = [meta["clean_name"] for meta in spec["indicators"].values()]
        wide_map: dict[tuple[str, int], dict] = {}
        for row in base_rows:
            if row["Objective"] != objective:
                continue
            key = (row["Country_Code"], int(row["Year"]))
            wide = wide_map.setdefault(key, {
                "Country_Name": row["Country_Name"],
                "Country_Code": row["Country_Code"],
                "Region": row["Region"],
                "Income_Group": row["Income_Group"],
                "Year": int(row["Year"]),
            })
            wide[row["Clean_Name"]] = row["Value"]
        wide_rows = sorted(wide_map.values(), key=lambda r: (str(r["Country_Name"]), int(r["Year"])))
        write_csv(OUTPUT_DIR / spec["output"], wide_rows, ["Country_Name", "Country_Code", "Region", "Income_Group", "Year", *metric_order])
        counts[spec["output"]] = len(wide_rows)
    return counts


def main() -> None:
    base_rows, audit_rows, cleaning_rows = build_base_dataset()
    write_csv(
        OUTPUT_DIR / "wdi_clean_base.csv",
        base_rows,
        ["Country_Name", "Country_Code", "Region", "Income_Group", "Year", "Objective", "Objective_Label", "Indicator_Code", "Indicator_Name", "Clean_Name", "Unit", "Value", "Was_Interpolated"],
    )
    write_csv(
        OUTPUT_DIR / "wdi_indicator_audit.csv",
        audit_rows,
        ["Objective", "Objective_Label", "Indicator_Code", "Clean_Name", "Confirmed_In_WDI", "Indicator_Name", "Topic", "Unit", "Analysis_Role"],
    )
    write_csv(
        OUTPUT_DIR / "wdi_cleaning_report.csv",
        cleaning_rows,
        ["Country_Code", "Country_Name", "Objective", "Indicator_Code", "Clean_Name", "Observed_Values", "Internal_Max_Missing_Gap", "Interpolated_Values", "Status"],
    )
    counts = build_objective_outputs(base_rows)
    print(f"Wrote data_output/wdi_clean_base.csv: {len(base_rows):,} long-format rows")
    for name, count in counts.items():
        print(f"Wrote data_output/{name}: {count:,} wide rows")
    print("Wrote data_output/wdi_indicator_audit.csv")
    print("Wrote data_output/wdi_cleaning_report.csv")

main()
